# Agrupamento por padrão de compra — Grocery Sales
Terceiro notebook do projeto (aprendizado **não supervisionado**). Ele:
1. Descreve a **distribuição das compras por categoria** (ECDF) e agrupa as categorias mais próximas com clustering hierárquico, dando um nome a cada grupo.
2. Testa o que realmente **diferencia os clientes** entre si, comparando cada característica com o que o acaso daria.
3. Agrupa os clientes pela única característica que passou no teste, o **tamanho da compra**, e dá nomes do dia a dia.

> Pré-requisito: rodar `limpeza.ipynb` para gerar a pasta `dados_limpos/`.

## 1. Carregar os dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

PASTA = Path("dados_limpos")
AZUL = "#2a6fdb"
SEMENTE = 42

categories = pd.read_csv(PASTA / "categories.csv")
products = pd.read_csv(PASTA / "products.csv", usecols=["ProductID", "Class", "Resistant"])
nomes = categories.set_index("CategoryID")["CategoryName"]
sales = pd.read_csv(PASTA / "sales.csv", usecols=["CustomerID", "SalesPersonID", "ProductID", "CategoryID", "Quantity", "Discount", "TotalPrice", "SalesDate"], parse_dates=["SalesDate"], dtype={"CustomerID": "int32", "SalesPersonID": "int32", "ProductID": "int32", "CategoryID": "int16", "Quantity": "int32", "Discount": "float32", "TotalPrice": "float32"})
sales["Categoria"] = sales.CategoryID.map(nomes)
print(f"{len(sales):,} vendas | {sales.CustomerID.nunique():,} clientes | {sales.CategoryID.nunique()} categorias")

## 2. Tabela cliente × categoria (número de compras)
Cada linha é um cliente e cada coluna uma categoria. O valor é **quantas vezes** o cliente comprou aquela categoria.

In [ ]:
contagem = sales.groupby(["CustomerID", "Categoria"]).size().unstack(fill_value=0)
print(contagem.shape)
display(contagem.head())

## 3. Como se distribuem as compras de cada categoria
No ECDF, cada linha é uma categoria: quanto mais à direita a curva, mais vezes os clientes compram aquela categoria. Curvas coladas indicam categorias com comportamento parecido.

In [ ]:
estatisticas = pd.DataFrame({"media": contagem.mean(), "mediana": contagem.median(), "desvio": contagem.std(), "p90": contagem.quantile(0.9), "sem_compra": (contagem == 0).mean()})
display(estatisticas.sort_values("media").round(2))

ordem_media = estatisticas.media.sort_values().index
cores = plt.cm.viridis(np.linspace(0, 1, len(ordem_media)))
fig, ax = plt.subplots(figsize=(9, 5))
for nome, cor in zip(ordem_media, cores):
    v = np.sort(contagem[nome].values)
    ax.step(v, np.arange(1, len(v) + 1) / len(v), where="post", color=cor, label=nome)
ax.set(title="ECDF do número de compras por categoria (legenda da menor para a maior média)", xlabel="Compras por cliente", ylabel="Fração acumulada")
ax.legend(ncol=2, fontsize=8); plt.show()

## 4. Quais categorias estão mais próximas
Cada categoria é resumida por cinco números da sua distribuição (as colunas da tabela acima), padronizados para pesarem igual. O **dendrograma** (método de Ward) junta primeiro as categorias mais parecidas.

Também testamos a outra ideia de proximidade: categorias que **os mesmos clientes compram juntas** (correlação entre colunas). Se a correlação fosse alta, valeria agrupar por ela.

In [ ]:
z = StandardScaler().fit_transform(estatisticas)
ligacao_cat = linkage(z, method="ward")

fig, ax = plt.subplots(figsize=(9, 4))
dendrogram(ligacao_cat, labels=list(estatisticas.index), ax=ax, color_threshold=0, above_threshold_color=AZUL)
ax.set(title="Dendrograma das categorias (Ward, sobre a forma da distribuição)", ylabel="Distância"); plt.show()

corr = contagem.corr().values
fora_da_diagonal = corr[~np.eye(len(corr), dtype=bool)]
print(f"Correlação entre categorias: mínima {fora_da_diagonal.min():.3f}, máxima {fora_da_diagonal.max():.3f}")
print("Alturas de junção:", np.round(ligacao_cat[:, 2], 2))

## 5. Fechar os grupos de categorias e dar nomes
O corte do dendrograma é `K_CATEGORIAS`. Os nomes ficam em `NOMES_GRUPOS`: use **qualquer categoria do grupo** como chave e o nome como valor. Sem nome, o grupo aparece como "Grupo A", "Grupo B"…

In [ ]:
K_CATEGORIAS = 4
NOMES_GRUPOS = {
    "Confections": "Campeão de vendas",
    "Meat": "Cesta do dia a dia",
    "Seafood": "Extras e mimos",
    "Grain": "Nicho da despensa",
}

lista_categorias = list(estatisticas.index)
rotulos = fcluster(ligacao_cat, K_CATEGORIAS, criterion="maxclust")
membros = {r: [c for c in ordem_media if rotulos[lista_categorias.index(c)] == r] for r in sorted(set(rotulos))}
nome_categoria_grupo = {}
for i, (r, cats) in enumerate(sorted(membros.items(), key=lambda kv: -estatisticas.media[kv[1]].mean())):
    nome_categoria_grupo[r] = next((NOMES_GRUPOS[c] for c in cats if c in NOMES_GRUPOS), f"Grupo {chr(65 + i)}")
    print(f"{nome_categoria_grupo[r]}: {', '.join(cats)} | média de compras por cliente: {estatisticas.media[cats].mean():.1f}")

mapa = {c: nome_categoria_grupo[rotulos[lista_categorias.index(c)]] for c in lista_categorias}
paleta = dict(zip(nome_categoria_grupo.values(), plt.cm.tab10.colors))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(ordem_media, estatisticas.media[ordem_media], color=[paleta[mapa[c]] for c in ordem_media])
ax.set(title="Média de compras por cliente, colorida por grupo de categorias", ylabel="Compras por cliente")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in paleta.values()], labels=list(paleta.keys())); plt.show()

## 6. O que diferencia os clientes? Teste contra o acaso
Se todos os clientes se comportassem igual, a parcela de compras em cada tipo de produto ainda variaria de cliente para cliente, só por sorte. Para cada característica comparamos o **desvio observado** entre clientes com o **desvio esperado só por acaso** (`razao = observado / esperado`). Razão perto de 1 significa que não há perfis de cliente naquela característica; razão bem acima de 1 indica estrutura real.

Por isso não usamos aqui o corte do ECDF no percentil 0,1 com a tabela 1/0 de "comprou ou não": a contagem por categoria tem um único pico (não é bimodal), e o corte só separa a cauda baixa das categorias de menor média (Grain, Dairy, Shell fish), criando grupos artificiais.

In [ ]:
vendas = sales.merge(products, on="ProductID", how="left")
cliente = vendas.CustomerID
indicadores = pd.DataFrame({
    "Produto de classe High": vendas.Class == "High",
    "Produto durável": vendas.Resistant == "Durable",
    "Compra com desconto": vendas.Discount > 0,
    "Compra no fim de semana": vendas.SalesDate.dt.dayofweek >= 5,
    "Compra pela manhã": vendas.SalesDate.dt.hour < 12,
    "Categoria animal": vendas.Categoria.isin(["Meat", "Poultry", "Seafood", "Shell fish"]),
    "Categoria vegetal": vendas.Categoria.isin(["Produce", "Cereals", "Grain"]),
    "Doces": vendas.Categoria == "Confections",
})
compras_medias = len(vendas) / cliente.nunique()
parcelas = indicadores.groupby(cliente.values).mean()
p = indicadores.mean()
teste = pd.DataFrame({"media": p, "desvio_observado": parcelas.std(), "desvio_acaso": np.sqrt(p * (1 - p) / compras_medias)})
teste["razao"] = teste.desvio_observado / teste.desvio_acaso
display(teste.round(4))

quantidade = vendas.groupby("CustomerID")["Quantity"].agg(["min", "max", "mean"])
print(f"Clientes com a mesma quantidade em todas as compras: {(quantidade['min'] == quantidade['max']).mean():.1%}")
print(f"Quantidade por compra: de {quantidade['mean'].min():.0f} a {quantidade['mean'].max():.0f}")

## 7. Grupos por tamanho de compra
Cada cliente compra sempre a **mesma quantidade** por compra (de 1 a 25 unidades), distribuída de forma uniforme entre os clientes. É a única característica do cliente que passou no teste, e explica quase toda a diferença de receita entre eles. Como a distribuição é uniforme e não tem "quinas", os cortes em `LIMITES` são uma escolha: quatro faixas de tamanho parecido.

In [ ]:
LIMITES = [0, 6, 12, 19, 25]
NOMES_TAMANHO = ["Compra de ocasião", "Compra de rotina", "Compra de família", "Atacadistas"]

qtd = quantidade["mean"].round()
tamanho = pd.cut(qtd, bins=LIMITES, labels=NOMES_TAMANHO).rename("Grupo")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(qtd, bins=np.arange(0.5, 26.5), color=AZUL, edgecolor="white")
for l in LIMITES[1:-1]:
    ax.axvline(l + 0.5, color="crimson", ls="--")
ax.set(title="Quantidade por compra de cada cliente", xlabel="Unidades por compra", ylabel="Clientes"); plt.show()

receita = vendas.groupby("CustomerID")["TotalPrice"].sum()
resumo = pd.DataFrame({
    "clientes": tamanho.value_counts(),
    "faixa_unidades": dict(zip(NOMES_TAMANHO, [f"{LIMITES[i] + 1} a {LIMITES[i + 1]}" for i in range(4)])),
    "compras_medias": contagem.sum(axis=1).groupby(tamanho, observed=True).mean().round(1),
    "receita_media": receita.groupby(tamanho, observed=True).mean().round(0),
}).loc[NOMES_TAMANHO]
resumo["percentual"] = (resumo.clientes / resumo.clientes.sum() * 100).round(1)
display(resumo)

por_grupo_cat = contagem.T.groupby(mapa).sum().T
mistura = por_grupo_cat.div(por_grupo_cat.sum(axis=1), axis=0).groupby(tamanho, observed=True).mean().loc[NOMES_TAMANHO]
display((mistura * 100).round(1).rename(columns=lambda c: f"{c} (%)"))

## 8. Simulações com outros K para os clientes
O K=4 da seção 7 foi uma escolha. Aqui testamos de 2 a 10 grupos de tamanho de compra com o K-Means e medimos:
- **Inércia** (cotovelo): quanto menor, mais homogêneos os grupos; sempre cai com mais grupos.
- **Silhouette**: separação entre grupos (−1 a 1). Em dados sem "quinas" costuma ficar parecida para qualquer K.
- **Davies-Bouldin** (menor é melhor) e **Calinski-Harabasz** (maior é melhor).
- **% da variação da receita explicada** pelos grupos: o quanto saber o grupo já diz da receita do cliente.
- **Gap statistic**: compara a inércia dos dados com a de dados uniformes sem grupos. Se a diferença é ~0, os dados não têm agrupamento natural.

In [ ]:
valores = qtd.values.reshape(-1, 1).astype(float)
rec = receita.loc[qtd.index].values
rng = np.random.RandomState(SEMENTE)
amostra_q = rng.choice(len(valores), 10000, replace=False)
amostra_gap = valores[rng.choice(len(valores), 20000, replace=False)]

def log_inercia(x, k):
    return np.log(KMeans(n_clusters=k, n_init=5, random_state=SEMENTE).fit(x).inertia_)

referencias = [rng.uniform(amostra_gap.min(), amostra_gap.max(), size=amostra_gap.shape) for _ in range(10)]
linhas = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMENTE).fit(valores)
    rot = km.labels_
    media_grupo = pd.Series(rec).groupby(rot).transform("mean").values
    ref = np.array([log_inercia(r, k) for r in referencias])
    linhas.append({"k": k, "inercia": km.inertia_, "silhouette": silhouette_score(valores[amostra_q], rot[amostra_q]), "davies_bouldin": davies_bouldin_score(valores, rot), "calinski_harabasz": calinski_harabasz_score(valores, rot), "receita_explicada_%": (1 - ((rec - media_grupo) ** 2).sum() / ((rec - rec.mean()) ** 2).sum()) * 100, "gap": ref.mean() - log_inercia(amostra_gap, k), "gap_erro": ref.std() * np.sqrt(1 + 1 / len(ref))})
sim_clientes = pd.DataFrame(linhas).set_index("k")
display(sim_clientes.round(3))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (col, titulo) in zip(axes, [("inercia", "Inércia (cotovelo)"), ("silhouette", "Silhouette"), ("receita_explicada_%", "% da receita explicada"), ("gap", "Gap statistic")]):
    ax.plot(sim_clientes.index, sim_clientes[col], "o-", color=AZUL); ax.set(title=titulo, xlabel="k")
axes[0].axvline(4, color="crimson", ls="--"); plt.tight_layout(); plt.show()

### Como ficam os grupos com K = 3, 4, 5 e 6
Se os dados tivessem um K natural, as faixas mudariam pouco em torno dele. Aqui os cortes simplesmente se redistribuem pela faixa de 1 a 25 unidades.

In [ ]:
for k in (3, 4, 5, 6):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMENTE).fit(valores)
    ordem = np.argsort(km.cluster_centers_.ravel())
    faixas = [f"{valores[km.labels_ == c].min():.0f} a {valores[km.labels_ == c].max():.0f} ({(km.labels_ == c).mean():.0%})" for c in ordem]
    print(f"K = {k}: " + " | ".join(faixas))

## 9. Simulações com outros K para as categorias
Mesma ideia para o dendrograma das categorias: cortar em 2 a 6 grupos e ver a silhouette (sobre as cinco medidas de distribuição padronizadas) e quem fica com quem.

In [ ]:
linhas = []
for k in range(2, 7):
    rot = fcluster(ligacao_cat, k, criterion="maxclust")
    grupos_k = {r: [c for c in ordem_media if rot[lista_categorias.index(c)] == r] for r in sorted(set(rot))}
    ordenados = sorted(grupos_k.values(), key=lambda cs: -estatisticas.media[cs].mean())
    linhas.append({"k": k, "silhouette": silhouette_score(z, rot), "salto_de_altura": ligacao_cat[-(k - 1), 2] - ligacao_cat[-k, 2] if k < 6 else np.nan, "grupos": " | ".join(", ".join(cs) for cs in ordenados)})
sim_categorias = pd.DataFrame(linhas).set_index("k")
pd.set_option("display.max_colwidth", 200)
display(sim_categorias.round(3))

## 10. Outros agrupamentos testados
Além do tamanho da compra, testamos as outras formas óbvias de agrupar. Para cada uma comparamos com o que o **acaso** daria:
- **Cidade do cliente** e **vendedor**: a fração da variação explicada pelos grupos (η²) contra a esperada só por sorte, `(grupos − 1) / (observações − 1)`.
- **Mistura de categorias** (o que o cliente compra): silhouette do K-Means contra clientes simulados ao acaso (cada cliente sorteia suas compras com as probabilidades globais das categorias).
- **Catálogo**: produtos por categoria, validade e preço.

In [ ]:
from scipy.stats import chi2_contingency

customers = pd.read_csv(PASTA / "customers.csv", usecols=["CustomerID", "CityID"]).set_index("CustomerID")
cli = pd.DataFrame({"quantidade": qtd, "receita": receita, "compras": contagem.sum(axis=1)}).join(customers)

def eta2(valor, grupo):
    g = valor.groupby(grupo)
    return (g.size() * (g.mean() - valor.mean()) ** 2).sum() / ((valor - valor.mean()) ** 2).sum()

linhas = []
esperado_cidade = (cli.CityID.nunique() - 1) / (len(cli) - 1)
for col in ["quantidade", "receita", "compras"]:
    linhas.append({"agrupamento": "Cidade do cliente", "variavel": col, "explicado": eta2(cli[col], cli.CityID), "esperado_acaso": esperado_cidade})
esperado_vend = (sales.SalesPersonID.nunique() - 1) / (len(sales) - 1)
for col in ["Quantity", "TotalPrice"]:
    linhas.append({"agrupamento": "Vendedor", "variavel": col, "explicado": eta2(sales[col].astype(float), sales.SalesPersonID), "esperado_acaso": esperado_vend})
cidade_categoria = contagem.join(customers).groupby("CityID").sum()
chi2, p_chi, _, _ = chi2_contingency(cidade_categoria.values)
display(pd.DataFrame(linhas).round(5))
print(f"Mistura de categorias por cidade: qui-quadrado p = {p_chi:.3f} (p alto = cidades compram a mesma mistura)")

In [ ]:
Xm = StandardScaler().fit_transform(contagem.values.astype(float))
idx = np.random.RandomState(SEMENTE).choice(len(Xm), 10000, replace=False)
reais = Xm[idx]
sim = np.random.default_rng(SEMENTE).multinomial(contagem.values.sum(axis=1)[idx], contagem.values.sum(axis=0) / contagem.values.sum())
embaralhada = StandardScaler().fit_transform(sim.astype(float))
linhas = []
for k in (2, 3, 4, 5):
    sil_real = silhouette_score(reais, KMeans(n_clusters=k, n_init=5, random_state=SEMENTE).fit(reais).labels_)
    sil_emb = silhouette_score(embaralhada, KMeans(n_clusters=k, n_init=5, random_state=SEMENTE).fit(embaralhada).labels_)
    linhas.append({"k": k, "silhouette_real": sil_real, "silhouette_acaso": sil_emb})
display(pd.DataFrame(linhas).set_index("k").round(3))

In [ ]:
catalogo = pd.read_csv(PASTA / "products.csv", usecols=["ProductID", "CategoryID", "Price", "VitalityDays"])
vendas_produto = sales.groupby("ProductID").size()
print(f"Vendas por produto: média {vendas_produto.mean():,.0f}, desvio {vendas_produto.std():.0f} | desvio esperado só por acaso {np.sqrt(vendas_produto.mean()):.0f}")

por_categoria = pd.DataFrame({"produtos": catalogo.groupby("CategoryID").size(), "vendas": sales.groupby("CategoryID").size()})
por_categoria.index = por_categoria.index.map(nomes)
print(f"Correlação entre nº de produtos da categoria e suas vendas: {por_categoria.produtos.corr(por_categoria.vendas):.3f}")
display(por_categoria.sort_values("produtos", ascending=False))

print(f"Produtos com prazo de validade: {(catalogo.VitalityDays > 0).mean():.0%} | sem prazo: {(catalogo.VitalityDays == 0).mean():.0%}")
print("Produtos por faixa de preço (R$ 0-10, 10-20, ...):", np.histogram(catalogo.Price, bins=10, range=(0, 100))[0].tolist())

### Placar dos agrupamentos

In [ ]:
r = pd.DataFrame(linhas)
placar = pd.DataFrame([
    ["Clientes por tamanho de compra", f"explica {sim_clientes.loc[4, 'receita_explicada_%']:.0f}% da receita (K=4)", "Real. Escolhido."],
    ["Clientes por cidade", f"explica {eta2(cli.receita, cli.CityID):.2%} da receita; o acaso daria {esperado_cidade:.2%}", "Acaso."],
    ["Clientes por vendedor", f"explica {eta2(sales.Quantity.astype(float), sales.SalesPersonID):.4%} da quantidade; o acaso daria {esperado_vend:.4%}", "Acaso."],
    ["Clientes por mistura de categorias", f"silhouette {r.silhouette_real.iloc[2]:.3f} contra {r.silhouette_acaso.iloc[2]:.3f} ao acaso (K=4)", "Acaso (artefato do K-Means)."],
    ["Clientes por horário, dia, desconto, classe do produto", "razão observado/acaso entre 1,005 e 1,010 (seção 6)", "Acaso."],
    ["Categorias por popularidade", f"correlação {por_categoria.produtos.corr(por_categoria.vendas):.2f} com o nº de produtos da categoria", "Real, mas reflete o tamanho do catálogo, não a preferência."],
    ["Produtos por validade ou preço", f"{(catalogo.VitalityDays > 0).mean():.0%} com prazo; preço uniforme de 0 a 100", "Real no catálogo, sem efeito no comportamento do cliente."],
], columns=["agrupamento", "evidência contra o acaso", "veredito"]).set_index("agrupamento")
pd.set_option("display.max_colwidth", 120)
display(placar)

## 11. Subgrupos por tipo de produto, dentro de cada tamanho de compra
Dentro de cada grupo de tamanho, tentamos separar os clientes pela mistura de produtos. Primeiro o teste: K-Means (K=3) sobre a parcela de compras em cada categoria, comparado com clientes **simulados ao acaso** (cada cliente sorteia suas compras com as probabilidades globais das categorias).

In [ ]:
rng_sub = np.random.default_rng(SEMENTE)
probs = contagem.values.sum(axis=0) / contagem.values.sum()
parcelas_cat = contagem.div(contagem.sum(axis=1), axis=0)
linhas = []
for nome_grupo in NOMES_TAMANHO:
    dentro = np.where(tamanho.loc[contagem.index].values == nome_grupo)[0]
    dentro = rng_sub.choice(dentro, 8000, replace=False)
    real = StandardScaler().fit_transform(parcelas_cat.values[dentro])
    sim_contagem = rng_sub.multinomial(contagem.values[dentro].sum(axis=1), probs)
    acaso = StandardScaler().fit_transform(sim_contagem / sim_contagem.sum(axis=1, keepdims=True))
    linhas.append({"grupo": nome_grupo, "silhouette_real": silhouette_score(real, KMeans(n_clusters=3, n_init=5, random_state=SEMENTE).fit(real).labels_), "silhouette_acaso": silhouette_score(acaso, KMeans(n_clusters=3, n_init=5, random_state=SEMENTE).fit(acaso).labels_)})
display(pd.DataFrame(linhas).set_index("grupo").round(3))

Silhouette real igual ao acaso: **não há subgrupos naturais**. Ainda assim, cada cliente tem uma **tendência** de cesta: o tipo de produto em que a parcela de compras mais passa da média geral (em desvios). Usamos três tipos (`TIPOS`) e damos a cada subgrupo o nome da cesta dominante.

> São subgrupos **descritivos**: a diferença entre eles é do tamanho da que o acaso produziria (tabela de comparação abaixo).

In [ ]:
TIPOS = {"Proteína animal": ["Meat", "Poultry", "Seafood", "Shell fish"], "Vegetais e grãos": ["Produce", "Cereals", "Grain"], "Doces": ["Confections"]}
SUBNOMES = {"Proteína animal": "Cesta de carnes", "Vegetais e grãos": "Cesta de vegetais", "Doces": "Cesta de doces"}

def tendencia(tabela_contagem):
    por_tipo = pd.DataFrame({t: tabela_contagem[cols].sum(axis=1) for t, cols in TIPOS.items()})
    parcela = por_tipo.div(tabela_contagem.sum(axis=1), axis=0)
    return parcela, ((parcela - parcela.mean()) / parcela.std()).idxmax(axis=1)

parcela_tipo, dominante = tendencia(contagem)
subgrupo = (tamanho.loc[contagem.index].astype(str) + " · " + dominante.map(SUBNOMES)).rename("Subgrupo")

resumo_sub = parcela_tipo.groupby([tamanho.loc[contagem.index], dominante], observed=True).mean().mul(100).round(1)
resumo_sub.insert(0, "clientes", subgrupo.groupby([tamanho.loc[contagem.index], dominante], observed=True).size())
resumo_sub.insert(1, "receita_media", receita.loc[contagem.index].groupby([tamanho.loc[contagem.index], dominante], observed=True).mean().round(0))
resumo_sub.index = resumo_sub.index.set_names(["tamanho", "cesta_dominante"])
display(resumo_sub)
print("Parcela média geral (%):", (parcela_tipo.mean() * 100).round(1).to_dict())

In [ ]:
sim_todos = rng_sub.multinomial(contagem.sum(axis=1).values, probs)
sim_tabela = pd.DataFrame(sim_todos, index=contagem.index, columns=contagem.columns)

def separacao(tabela_contagem):
    parcela, dom = tendencia(tabela_contagem)
    return np.mean([parcela.loc[dom == t, t].mean() - parcela[t].mean() for t in TIPOS]) * 100

print(f"Quanto a parcela do tipo dominante fica acima da média no seu subgrupo (pontos percentuais): real {separacao(contagem):.2f} | simulado ao acaso {separacao(sim_tabela):.2f}")

## 12. Salvar

In [ ]:
clientes_grupos = tamanho.to_frame().assign(QuantidadePorCompra=qtd.astype(int), Receita=receita.round(2), Cesta=dominante.map(SUBNOMES), Subgrupo=subgrupo).reset_index()
saida = PASTA / "clientes_grupos.csv"
clientes_grupos.to_csv(saida, index=False)
pd.Series(mapa, name="GrupoCategoria").rename_axis("Categoria").reset_index().to_csv(PASTA / "categorias_grupos.csv", index=False)
print(f"salvo: {saida} ({len(clientes_grupos):,} clientes) e categorias_grupos.csv")

## 13. Conclusões
- **Categorias**: não andam juntas nas compras dos clientes (correlações perto de zero), então o agrupamento que funciona é o da **forma da distribuição**. Ele separa quatro faixas: *Campeão de vendas* (Confections, a mais comprada), *Cesta do dia a dia* (Cereals, Meat, Poultry, Produce), *Extras e mimos* (Shell fish, Dairy, Beverages, Seafood, Snails) e *Nicho da despensa* (Grain, a menos comprada).
- **Clientes**: classe do produto, durabilidade, desconto, dia da semana, horário e a parcela gasta em produtos animais, vegetais e doces variam entre clientes como o acaso faria (razão perto de 1). Por isso **não há** perfis como carnívoros, vegetarianos ou premium nestes dados, e o K-Means sobre a tabela 1/0 só encontraria artefatos do corte.
- **Tamanho da compra**: é a única característica real. Cada cliente compra sempre a mesma quantidade (1 a 25 unidades), o que gera os grupos *Compra de ocasião*, *Compra de rotina*, *Compra de família* e *Atacadistas*, com receita média crescente e a mesma mistura de categorias.
- **Qual K para os clientes** (seção 8): não existe K natural. O gap statistic é negativo para todo K, e a silhouette tem formato de "U" (alta em K=2 e K=10), o que é um artefato de dados em uma só dimensão e distribuição uniforme. O critério útil é a **receita explicada**: 69% (K=2), 82% (K=3), 87% (K=4), 89% (K=5), 90% (K=6) e menos de 1 ponto por grupo a mais depois disso. **K=4 é um bom ponto de equilíbrio** (passar de 4 para 5 ganha só ~2 pontos), mas é uma escolha prática, e K=3 (36/32/32%) seria igualmente defensável se quiser menos grupos.
- **Qual K para as categorias** (seção 9): a silhouette cai de 0,57 (K=2) para 0,56 (K=3), 0,50 (K=4), 0,43 (K=5) e 0,38 (K=6). O maior salto de altura no dendrograma está em K=2 (populares contra menos populares), seguido de K=3 (isola Grain) e K=4 (isola Confections) quase empatados; em K=5 o salto some. **K=3 é o mais sustentado pelos números; K=4 é a divisão mais fina que ainda se justifica** e foi mantida por deixar a categoria mais comprada em destaque.
- **Outros agrupamentos** (seção 10): cidade, vendedor, mistura de categorias, horário, dia, desconto e classe do produto não separam nada além do acaso. A mistura de categorias dá silhouette do K-Means igual ao de clientes simulados ao acaso.
- **Os grupos de categorias refletem o catálogo, não a preferência**: cada produto vende quase o mesmo (14.802 vendas em média, desvio 113 contra 122 esperado pelo acaso), então as vendas de uma categoria são proporcionais ao número de produtos dela (correlação 1,00; Confections tem 57 produtos e Grain 28).
- **Subgrupos por tipo de produto** (seção 11): dentro de cada tamanho de compra a mistura de produtos é igual à de clientes simulados ao acaso (silhouette 0,063 nos dois). Os subgrupos *Cesta de carnes*, *Cesta de vegetais* e *Cesta de doces* são **descritivos**: a parcela do tipo dominante fica 5,02 pontos percentuais acima da média no subgrupo, exatamente o que o acaso também produz (5,02), e a receita média é a mesma entre os subgrupos do mesmo tamanho.
- **Veredito**: o agrupamento que mais faz sentido é o **tamanho da compra** dos clientes. É o único que os dados sustentam como comportamento do cliente.
- O conjunto desses resultados é consistente com o sinal de dados sintéticos já apontado em `limpeza.ipynb`.